In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

In [ ]:
import torch

from tqdm import tqdm

from src.dataset import ALL_CLASSES, SpeechCommandsDatasetFactory
from src.model import MelResNetKWS
import pandas as pd

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
factory = SpeechCommandsDatasetFactory("../test")

dataset = factory.all()

In [5]:
from torch.utils.data import DataLoader

batch_size = 256

loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
)

In [21]:
best_state = torch.load("../distillated.pt", map_location="cpu")

model = MelResNetKWS(num_classes=12)

model.backbone.load_state_dict(best_state["model_a_backbone"])
model.head.load_state_dict(best_state["head"])

model = model.to(DEVICE)

model.eval()

MelResNetKWS(
  (backbone): Sequential(
    (preprocess): InputPreprocess()
    (mel): MelSpectrogram(
      (spectrogram): Spectrogram()
      (mel_scale): MelScale()
    )
    (db): AmplitudeToDB()
    (norm): SpecNormalize()
    (unsqueeze): Unsqueeze()
    (stem): Sequential(
      (0): Conv2d(1, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01)
    )
    (stage1): Sequential(
      (0): ResBlock2D(
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act1): LeakyReLU(negative_slope=0.01)
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act2): LeakyReLU(negative_slope=0.01)
        (dropout): Dropout2d(p=0.05, inplace=False)
        (con

In [19]:
results = {"fname": [], "label": []}

pbar = tqdm(loader, desc="Inference", leave=False)

for batch in pbar:
    speech = batch["speech"].to(DEVICE, non_blocking=True)
    path = batch["path"]

    with torch.no_grad():
        logits = model(speech)
        labels = logits.cpu().numpy().argmax(axis=1)
        class_labels = [ALL_CLASSES[label] for label in labels]

    results["fname"].extend(path)
    results["label"].extend(class_labels)

In [20]:
df = pd.DataFrame(results)

df.to_csv("../submission.csv", index=False)